<a href="https://colab.research.google.com/github/muskan-1301/speak_with_pdf/blob/main/Module_3_Assignment_OpenRouter_completed_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install dependencies (run once)
!pip install -q pypdf openai numpy



# Speak with PDF Assignment

**Assignment type:** RAG mini-app with three required functions  
**File to submit:** `pdf_ai.ipynb` (the notebook produced from this spec)  
**Language:** Python 3.x (Codio / Jupyter)  
**Time estimate:** 2–3 hours

---

## Objective

Build a **“Speak with PDF” RAG pipeline** that runs inside a Jupyter notebook and is compatible with **NBGrader**. You will implement **three functions**:

1) `parse_pdf_to_chunks(pdf_path, chunk_size=800, overlap=120) -> List[str]`  
2) `embed_and_index(chunks, use_pinecone=True, model="text-embedding-3-small") -> dict`  
3) `answer_query(question, index_obj, top_k=3, gpt_model="gpt-4o-mini") -> str`

**Grading will use OpenAI keys.** Learners must provide valid `OPENAI_API_KEY` (and Pinecone credentials if using Pinecone).

---

## What you will build

A tiny RAG system that:
- loads a PDF,
- extracts and **chunks** text,
- builds a **vector index** from OpenAI embeddings (Pinecone recommended; in‑memory/FAISS acceptable),
- answers a natural-language query by retrieving top‑k relevant chunks and generating an answer with an OpenAI chat model.

This mirrors production flow (PDF → chunks → embeddings → index → retrieval → answer).


---

## Tools & Libraries

- `openai` (for embeddings + chat completions)
- `pypdf` **or** `pdfplumber` (choose one) for text extraction
- Vector store: **Pinecone** (recommended) via `pinecone-client`; you may alternatively use **FAISS** or a simple in‑memory index for development
- `numpy`, `re`, `math`, `hashlib` (as needed)

> You may import additional libraries if helpful. Use your judgment.

**Environment variables expected (for grading):**
- `OPENAI_API_KEY`
- If using Pinecone: `PINECONE_API_KEY`, `PINECONE_ENV` (or project config), and an index name you create in notebook (e.g., `pdf-mini-app`).



---

## Function Requirements
You'll have to implment the following functionality down below

### 1) `parse_pdf_to_chunks(pdf_path, chunk_size=800, overlap=120) -> list[str]`

**Inputs**
- `pdf_path: str` — path to a readable PDF file (e.g., `data/sample.pdf`)
- `chunk_size: int` (default 800) — approximate number of characters per chunk
- `overlap: int` (default 120) — number of overlapping characters between adjacent chunks

**Output**
- `List[str]` — ordered, non-empty chunks of plain text. No chunk should exceed `chunk_size + 20` characters (allowing minor boundary variance).

**Behavioral constraints**
- Strip excessive whitespace and normalize newlines.
- Deterministic chunk boundaries given the same inputs.
- Apply overlap between chunks (except the first).

**Recommended approach**
1. Extract all PDF text into a single string using `pypdf` **or** `pdfplumber`.
2. Normalize whitespace.
3. Greedy chunk by character count with the requested overlap.



---

### 2) `embed_and_index(chunks, use_pinecone=True, model="text-embedding-3-small") -> dict`

**Purpose**  
Create embeddings with **OpenAI** and store them in a vector index (Pinecone preferred).

**Inputs**
- `chunks: List[str]` — the output of `parse_pdf_to_chunks`
- `use_pinecone: bool` — if `True`, upsert to Pinecone; if `False`, store locally (FAISS or simple in‑memory array search)
- `model: str` — OpenAI embedding model name (e.g., `"text-embedding-3-small"`)

**Output**
- `dict` with fields (minimum):
  - `"chunks"`: `List[str]`
  - `"index_backend"`: `"pinecone"` or `"local"`
  - `"index_ref"`: backend-specific handle (e.g., Pinecone index name / namespace, or local matrix + ids)
  - `"embed_model"`: embedding model used
  - `"dim"`: embedding dimensionality

**Notes**
- When `use_pinecone=True`: create (if needed) and upsert vectors with stable IDs (e.g., `doc-0`, `doc-1`, …). Keep a mapping from ID → original chunk text in the dict.
- When `use_pinecone=False`: store embeddings in memory (e.g., NumPy matrix) along with their chunk IDs for cosine‑similarity search.



---

### 3) `answer_query(question, index_obj, top_k=3, gpt_model="gpt-4o-mini") -> str`

**Purpose**  
Retrieve relevant chunks and generate an answer using an OpenAI chat model.

**Inputs**
- `question: str`
- `index_obj: dict` — the output from `embed_and_index`
- `top_k: int` — number of chunks to retrieve
- `gpt_model: str` — OpenAI chat model (e.g., `"gpt-4o-mini"`)

**Output**
- `str` — the final answer.

**Required behavior**
- Embed the **question** using the **same embedding model** used to build the index.
- Retrieve **top_k** chunks by cosine similarity (Pinecone query or local similarity search).
- Construct a concise prompt:
  - System: “You are a helpful assistant. Use ONLY the provided context to answer. If not answerable from context, respond: ‘I don’t know based on the document.’”
  - User: include the `question` and a `Context:` section concatenating the retrieved chunks (with simple separators).
- Call the OpenAI chat completion endpoint and **return the assistant’s message string**.



In [2]:
# This cell may be run by the grader
import os, re, math, hashlib
import numpy as np

In [3]:
import os
from getpass import getpass

# Enter your OpenRouter key when prompted (never hard-code it in the notebook)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or getpass("Enter your OpenRouter API key: ")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"


Enter your OpenRouter API key: ··········


In [4]:
from openai import OpenAI

assert OPENROUTER_API_KEY, "Please set OPENROUTER_API_KEY"
# Kept for graders that read this name. Requests go to OpenRouter, not api.openai.com.
OPENAI_API_KEY = OPENROUTER_API_KEY
USE_PINECONE = False  # local in-memory index (allowed by the spec)

def get_client():
    """OpenAI-compatible client pointed at OpenRouter."""
    return OpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL)

def or_model(name: str) -> str:
    """OpenRouter model IDs look like 'openai/gpt-4o-mini'. Add the prefix if missing."""
    return name if "/" in name else f"openai/{name}"


<center><h1>--------------- Student Code ---------------</h1></center>

### B. Implement `parse_pdf_to_chunks`  **(student solution cell)**

### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [5]:
from pypdf import PdfReader
from typing import List

def parse_pdf_to_chunks(pdf_path: str, chunk_size: int = 800, overlap: int = 120) -> List[str]:
    """Extract text from a PDF, normalize whitespace and split it into overlapping chunks."""
    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError("Need chunk_size > 0 and 0 <= overlap < chunk_size")

    reader = PdfReader(pdf_path)
    # Empty or image-only pages return None, so fall back to ""
    pages = [(page.extract_text() or "") for page in reader.pages]
    text = "\n\n".join(pages)

    # Normalize newlines and whitespace
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t\f\v]+", " ", text)      # collapse runs of spaces/tabs
    text = re.sub(r" *\n *", "\n", text)          # trim spaces around newlines
    text = re.sub(r"\n{3,}", "\n\n", text).strip()  # at most one blank line
    if not text:
        return []

    # Greedy character chunks; each new chunk starts (chunk_size - overlap) after the last
    step = chunk_size - overlap
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)
        if end == len(text):
            break
        start += step
    return chunks


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `parse_pdf_to_chunks`.
 -  If the function is not found, you answer will be rejected by the system

### C. Implement `embed_and_index`  **(student solution cell)**

### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [6]:
import numpy as np

def embed_texts(texts: List[str], model: str) -> np.ndarray:
    """Embed a list of texts through OpenRouter in batches of 100."""
    client = get_client()
    vectors = []
    for i in range(0, len(texts), 100):
        resp = client.embeddings.create(model=or_model(model), input=texts[i:i + 100])
        vectors.extend(d.embedding for d in resp.data)
    return np.array(vectors, dtype=np.float32)

def _normalize_rows(M: np.ndarray) -> np.ndarray:
    """Scale each row to unit length so a dot product equals cosine similarity."""
    norms = np.linalg.norm(M, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return M / norms

def embed_and_index(chunks, use_pinecone=True, model="text-embedding-3-small") -> dict:
    """Embed chunks and build an index. Uses the local backend (Pinecone is not set up here)."""
    if not chunks:
        raise ValueError("No chunks to index. Check that the PDF has extractable text.")
    if use_pinecone:
        print("Pinecone is not configured in this notebook, so the local index is used.")

    ids = [f"doc-{i}" for i in range(len(chunks))]   # stable IDs
    M = _normalize_rows(embed_texts(list(chunks), model))
    return {
        "chunks": list(chunks),
        "index_backend": "local",
        "index_ref": {"matrix": M, "ids": ids},
        "embed_model": model,
        "dim": int(M.shape[1]),
        "id_to_chunk": dict(zip(ids, chunks)),
    }


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `embed_and_index`.
 -  If the function is not found, you answer will be rejected by the system

### D. Implement `answer_query`  **(student solution cell)**

### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [7]:
SYSTEM_PROMPT = (
    "You are a helpful assistant. Use ONLY the provided context to answer. "
    "If not answerable from context, respond: 'I don't know based on the document.'"
)

def retrieve(question: str, index_obj: dict, top_k: int = 3):
    """Return the top_k (id, score, text) chunks by cosine similarity."""
    q = _normalize_rows(embed_texts([question], index_obj["embed_model"]))[0]  # same model as the index
    M = index_obj["index_ref"]["matrix"]
    ids = index_obj["index_ref"]["ids"]
    scores = M @ q
    # Sort by score (high first); ties go to the earlier chunk so results are stable
    order = sorted(range(len(ids)), key=lambda i: (-scores[i], i))[:top_k]
    return [(ids[i], float(scores[i]), index_obj["id_to_chunk"][ids[i]]) for i in order]

def answer_query(question: str, index_obj: dict, top_k: int = 3, gpt_model: str = "gpt-4o-mini") -> str:
    """Retrieve relevant chunks, then ask the chat model to answer from them only."""
    hits = retrieve(question, index_obj, top_k=top_k)
    context = "\n\n---\n\n".join(text for _, _, text in hits)
    user_msg = f"Question: {question}\n\nContext:\n{context}"
    resp = get_client().chat.completions.create(
        model=or_model(gpt_model),
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0,
        max_tokens=400,
    )
    return (resp.choices[0].message.content or "").strip()


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `answer_query`.
 -  If the function is not found, you answer will be rejected by the system


### E. (Optional) Demo cell (ungraded)
Provide a small demo that:
- Loads `data/sample.pdf`
- Runs all three functions
- Prints an example answer



In [8]:
# Demo: load a PDF, index it, ask one question
PDF_PATH = "/content/sample_data/Northwind_Remote_Work_Policy.pdf"   # set to your uploaded PDF path

chunks = parse_pdf_to_chunks(PDF_PATH, chunk_size=800, overlap=120)
print("chunks:", len(chunks), "| longest chunk:", max(map(len, chunks)), "chars")

index_obj = embed_and_index(chunks, use_pinecone=USE_PINECONE)
print("indexed with", index_obj["embed_model"], "| dim:", index_obj["dim"])
print(answer_query("What is the main topic of this document?", index_obj, top_k=3))

# Required playtests: 3 domain questions, vanilla LLM vs RAG answer
QUESTIONS = [
    "How much is the home office allowance and when must receipts be submitted?",
    "What happens if an employee loses a laptop and reports it late?",
    "How long must an employee work for the company before becoming eligible for remote work?",
]
assert all(not q.startswith("REPLACE") for q in QUESTIONS), "Replace the placeholder questions first"

def vanilla_answer(question: str, gpt_model: str = "gpt-4o-mini") -> str:
    """Answer with no PDF context, as a baseline."""
    resp = get_client().chat.completions.create(
        model=or_model(gpt_model),
        messages=[{"role": "user", "content": question}],
        temperature=0,
        max_tokens=400,
    )
    return (resp.choices[0].message.content or "").strip()

for q in QUESTIONS:
    print("Q:", q)
    print("VANILLA:", vanilla_answer(q))
    print("RAG:", answer_query(q, index_obj, top_k=3))
    print("-" * 60)


chunks: 4 | longest chunk: 800 chars
indexed with text-embedding-3-small | dim: 1536
The main topic of this document is the policy regarding remote work and equipment for employees at Northwind Analytics. It outlines eligibility, core hours, equipment provided, and reimbursement for expenses related to remote work.
Q: How much is the home office allowance and when must receipts be submitted?
VANILLA: The home office allowance can vary depending on the country and specific tax regulations in place. In the UK, for example, employees can claim a flat rate of £6 per week for working from home, which does not require receipts. However, if you incur higher costs, you may be able to claim actual expenses, which would require receipts and detailed records.

In the US, the home office deduction allows self-employed individuals to deduct expenses related to the portion of their home used for business. The specifics can vary widely based on individual circumstances.

As for the submission of rece

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  If the function is not found, you answer will be rejected by the system

---

## Hidden Tests (run by grader)

Hidden tests will verify that:

- `parse_pdf_to_chunks` applies overlap correctly, is deterministic, and honors size bounds.
- `embed_and_index` calls OpenAI embeddings and builds a valid index (Pinecone or local), with stable IDs and correct dimensionality/shape.
- Retrieval uses cosine similarity and returns the expected top‑k order on crafted inputs.
- `answer_query` embeds the question with the same model used for chunks, constructs a context‑bounded prompt, calls OpenAI chat completions, and returns a clean string.
- No hard‑coded secrets; keys must be read from environment.
- Notebook runs top‑to‑bottom reproducibly when keys are present.



---

## Rubric (100 pts)

- **PDF parsing & chunking (30 pts)**
  - Correct extraction and normalization (10)
  - Deterministic overlapping chunks within size bounds (15)
  - Edge cases (empty pages, weird whitespace) (5)

- **Embeddings & indexing (35 pts)**
  - Correct OpenAI embedding usage and shapes (15)
  - Proper index structure + metadata (10)
  - Stable IDs and retrievability (10)

- **Query flow & answer (25 pts)**
  - Correct cosine similarity retrieval and stable tie‑break (10)
  - Prompt enforces context‑only answering; sensible output (15)

- **Code quality & constraints (10 pts)**
  - Clean, readable, documented code (5)
  - Secure key handling; graceful error handling (5)



---

## (Optional) Pinecone/OpenAI Notes

- For Pinecone: create an index (e.g., 384 or 1536 dim depending on embedding model) and upsert vectors with metadata (`{"chunk": "...", "order": i}`).
- For local dev without Pinecone: store embeddings in a NumPy matrix and do cosine similarity. Grading is OK with either backend as long as OpenAI embeddings + chat are used.
- Keep your mapping from vector IDs to original chunk text; you’ll need it for context construction.



---

## Submission

Submit your completed notebook. Ensure it runs top‑to‑bottom without modification in a clean environment **with valid API keys set**. The grader will run hidden tests using OpenAI for embeddings and chat.


---

<h2><center> Completing your test </center></h2>

- Once you have completed your test and answered all the questions:
  
- You can open the guide using the button located on the far right of your screen.

- The button will look like this image:  
  ![](https://github.com/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%203/additional_files/End_guide_button.png?raw=1)

- Clicking on this button will allow you to mark the test as complete using the button shown below:  
  ![](https://github.com/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%203/additional_files/completion_button.png?raw=1)

- Please click on **Mark as Completed** to end and submit your test.

- Thank you!
